# exp001_baseline train

Notebook-first CV evaluation for the deterministic last-known TVT baseline.

## Contents

1. Setup and configuration
2. Metric and data helpers
3. CV scoring helpers
4. Well-level CV run
5. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from math import sqrt
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from baseline import HORIZONTAL_SUFFIX, predict_from_prefix, primary_strategy, well_id_from_path
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
primary = primary_strategy(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Output root:", paths.output_root)
print("Artifacts:", paths.artifacts_dir)
print("Primary strategy:", primary)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)

## 2. Metric and data helpers


In [ ]:
def finite_float(value: float | None, digits: int = 6) -> float | None:
    if value is None or not np.isfinite(value):
        return None
    return round(float(value), digits)


def rmse_from_sums(sse: float, count: int) -> float | None:
    if count <= 0:
        return None
    return sqrt(sse / count)


def rmse_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> float | None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return None
    residual = y_pred[valid] - y_true[valid]
    return float(np.sqrt(np.mean(residual * residual)))


def train_files(paths: ExperimentPaths, debug: bool, max_wells: int | None) -> list[Path]:
    files = sorted(paths.train_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no train horizontal well CSVs found in {paths.train_data_dir}")
    if debug:
        limit = max_wells
        if limit is None:
            limit = int(config.get("runtime", {}).get("debug_n_wells", 30))
        files = files[:limit]
    elif max_wells is not None:
        files = files[:max_wells]
    return files


def build_fold_map(files: list[Path], n_folds: int) -> tuple[dict[str, int], int]:
    well_ids = [well_id_from_path(path) for path in files]
    n_splits = min(max(1, n_folds), len(well_ids))
    if n_splits < 2:
        return {well_id: 0 for well_id in well_ids}, 1

    fold_map: dict[str, int] = {}
    x = np.arange(len(well_ids)).reshape(-1, 1)
    groups = np.asarray(well_ids)
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (_, valid_idx) in enumerate(splitter.split(x, groups=groups)):
        for index in valid_idx:
            fold_map[well_ids[int(index)]] = fold
    return fold_map, n_splits

## 3. CV scoring helpers


In [ ]:
def add_score(
    stats: dict[str, Any],
    strategy: str,
    fold: int,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return
    residual = y_pred[valid] - y_true[valid]
    sse = float(np.sum(residual * residual))
    count = int(valid.sum())
    stats[strategy]["fold_sse"][fold] += sse
    stats[strategy]["fold_n"][fold] += count
    stats[strategy]["total_sse"] += sse
    stats[strategy]["total_n"] += count
    stats[strategy]["well_rmse"].append(float(np.sqrt(sse / count)))


def summarize_strategy(strategy_stats: dict[str, Any]) -> dict[str, Any]:
    fold_rmse = [
        finite_float(rmse_from_sums(sse, int(count)))
        for sse, count in zip(strategy_stats["fold_sse"], strategy_stats["fold_n"], strict=True)
    ]
    valid_fold_rmse = [value for value in fold_rmse if value is not None]
    well_rmse = np.asarray(strategy_stats["well_rmse"], dtype=float)
    return {
        "oof_rmse": finite_float(
            rmse_from_sums(strategy_stats["total_sse"], int(strategy_stats["total_n"]))
        ),
        "mean_fold_rmse": finite_float(
            float(np.mean(valid_fold_rmse)) if valid_fold_rmse else None
        ),
        "fold_rmse": fold_rmse,
        "rows": int(strategy_stats["total_n"]),
        "per_well_rmse_mean": finite_float(float(np.mean(well_rmse)) if well_rmse.size else None),
        "per_well_rmse_median": finite_float(
            float(np.median(well_rmse)) if well_rmse.size else None
        ),
    }


def write_csv_artifacts(
    paths: ExperimentPaths,
    well_records: list[dict[str, Any]],
) -> None:
    if well_records:
        pd.DataFrame(well_records).to_csv(paths.artifacts_dir / "well_metrics.csv", index=False)

## 4. Well-level CV run


In [ ]:
files = train_files(paths, debug=DEBUG, max_wells=MAX_WELLS)
fold_map, n_splits = build_fold_map(files, int(config["validation"]["n_folds"]))

strategy_list = config["model"]["strategies"]
stats = {
    strategy: {
        "fold_sse": np.zeros(n_splits, dtype=float),
        "fold_n": np.zeros(n_splits, dtype=np.int64),
        "total_sse": 0.0,
        "total_n": 0,
        "well_rmse": [],
    }
    for strategy in strategy_list
}
well_records: list[dict[str, Any]] = []

for file_index, path in enumerate(files, start=1):
    well_id = well_id_from_path(path)
    fold = fold_map[well_id]
    df = pd.read_csv(path)
    prediction = predict_from_prefix(df, config)
    y_true = df.loc[prediction.eval_indices, config["data"]["target_column"]].to_numpy(dtype=float)

    record: dict[str, Any] = {
        "well_id": well_id,
        "fold": fold,
        "n_rows": int(len(df)),
        "n_known": int(prediction.last_known_index + 1),
        "n_eval": int(prediction.eval_indices.size),
        "last_known_index": prediction.last_known_index,
        "last_known_md": prediction.last_known_md,
        "last_known_tvt": prediction.last_known_tvt,
        "recent_slope": prediction.recent_slope,
    }
    for strategy, y_pred in prediction.predictions.items():
        add_score(stats, strategy, fold, y_true, y_pred)
        record[f"{strategy}_rmse"] = finite_float(rmse_from_arrays(y_true, y_pred))
    well_records.append(record)

    if file_index % 100 == 0:
        print(f"Processed {file_index}/{len(files)} wells")

fold_records: list[dict[str, Any]] = []
strategies = {strategy: summarize_strategy(strategy_stats) for strategy, strategy_stats in stats.items()}
for strategy, strategy_stats in stats.items():
    for fold in range(n_splits):
        fold_records.append(
            {
                "strategy": strategy,
                "fold": fold,
                "rmse": finite_float(
                    rmse_from_sums(strategy_stats["fold_sse"][fold], int(strategy_stats["fold_n"][fold]))
                ),
                "rows": int(strategy_stats["fold_n"][fold]),
            }
        )

write_csv_artifacts(paths, well_records)

## 5. Metrics and artifacts


In [ ]:
primary_metrics = strategies[primary]
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": "debug_completed" if DEBUG else "completed",
    "created_at": config.get("experiment", {}).get("created_at"),
    "updated_at": datetime.now(UTC).isoformat(),
    "debug": DEBUG,
    "cv": primary_metrics["oof_rmse"],
    "cv_mean_fold_rmse": primary_metrics["mean_fold_rmse"],
    "public_lb": None,
    "private_lb": None,
    "metric": config.get("validation", {}).get("metric"),
    "seed": config.get("validation", {}).get("seed"),
    "primary_strategy": primary,
    "n_folds": n_splits,
    "n_wells": len(files),
    "strategies": strategies,
    "key_idea": config.get("experiment", {}).get("description"),
    "notes": (
        "Leak-safe deterministic prefix baseline. "
        "Submission uses last-known TVT anchor by default."
    ),
}
paths.metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

print("CV RMSE:", metrics["cv"])
print("Metrics written:", paths.metrics_path)